In [136]:
import tensorflow as tf
from tensorflow.keras.layers import Dense, Input, Embedding, Dropout, LayerNormalization, TextVectorization
from tensorflow.keras.models import Model
import numpy as np

## Defining Positional Encoding

In [137]:
def positional_encoding(position, d_model):
    angle_rads = np.arange(position)[:, np.newaxis] / np.power(
        10000, (2 * (np.arange(d_model) // 2)) / np.float32(d_model)
    )
    angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])  # apply sin to even indices
    angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])  # apply cos to odd indices
    pos_encoding = angle_rads[np.newaxis, ...]
    return tf.cast(pos_encoding, dtype=tf.float32)

## Defining Multi-head Attention 

In [138]:
class MultiHeadAttention(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.d_model = d_model

        assert d_model % self.num_heads == 0

        self.depth = d_model // self.num_heads

        self.wq = Dense(d_model)
        self.wk = Dense(d_model)
        self.wv = Dense(d_model)
        self.dense = Dense(d_model)

    def split_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.depth))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def scaled_dot_product_attention(self, q, k, v, mask):
              matmul_qk = tf.matmul(q, k, transpose_b=True)
              dk = tf.cast(tf.shape(k)[-1], tf.float32)
              scaled_attention_logits = matmul_qk / tf.math.sqrt(dk)
          
              if mask is not None:
                  scaled_attention_logits += (mask * -1e9)
          
              attention_weights = tf.nn.softmax(scaled_attention_logits, axis=-1)
              output = tf.matmul(attention_weights, v)
              return output, attention_weights
    
    def call(self, v, k, q, mask):
                batch_size = tf.shape(q)[0]
                q = self.wq(q)
                k = self.wk(k)
                v = self.wv(v)
                q = self.split_heads(q, batch_size)
                k = self.split_heads(k, batch_size)
                v = self.split_heads(v, batch_size)
                
                attention, attention_weights = self.scaled_dot_product_attention(q, k, v, mask)
                attention = tf.transpose(attention, perm=[0, 2, 1, 3])
                attention = tf.reshape(attention, (batch_size, -1, self.d_model))
                output = self.dense(attention)
                return output

## Defining Feed Forward Network


In [139]:
class PositionwiseFeedforward(tf.keras.layers.Layer):
    def __init__(self, d_model, dff):
        super().__init__()
        self.dense1 = Dense(dff, activation='relu')
        self.dense2 = Dense(d_model)

    def call(self, x):
        x = self.dense1(x)
        return self.dense2(x)

## Defining Transformer Block


In [140]:
class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads, dff, dropout_rate=0.1):
        super().__init__()
        self.att = MultiHeadAttention(d_model, num_heads)
        self.ffn = PositionwiseFeedforward(d_model, dff)
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(dropout_rate)
        self.dropout2 = Dropout(dropout_rate)

    def call(self, x, training=False, mask=None):
        attn_output = self.att(x, x, x, mask=mask)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(x + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        out2 = self.layernorm2(out1 + ffn_output)
        return out2

## Encoder

In [141]:
class Encoder(tf.keras.layers.Layer):
    def __init__(self, num_layers, d_model, num_heads, dff, input_vocab_size, maximum_position_encoding, dropout_rate=0.1):
        super(Encoder, self).__init__()
        self.d_model = d_model
        self.num_layers = num_layers
        
        self.embedding = Embedding(input_vocab_size, d_model)
        self.pos_encoding = positional_encoding(maximum_position_encoding, d_model)
        self.dropout = Dropout(dropout_rate)
        self.enc_layers = [TransformerBlock(
            d_model, num_heads, dff, dropout_rate) for _ in range(num_layers)
            ]

    def call(self, x, training, mask):
        seq_len = tf.shape(x)[1]

        x = self.embedding(x)
        x += self.pos_encoding[:, :seq_len, :]
        x = self.dropout(x, training=training)
        # for i in range(self.num_layers):
        #     x = self.enc_layers[i](x, training=training, mask=mask)
        for layer in self.enc_layers:
            x = layer(x, training=training, mask=mask)
            
        return x

## Decoder Block

In [142]:
class DecoderBlock(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads, dff, dropout_rate=0.1):
        super().__init__()
        self.mha1 = MultiHeadAttention(d_model, num_heads)
        
        self.mha2 = MultiHeadAttention(d_model, num_heads)

        self.ffn = PositionwiseFeedforward(d_model, dff)

        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.layernorm3 = LayerNormalization(epsilon=1e-6)

        self.dropout1 = Dropout(dropout_rate)
        self.dropout2 = Dropout(dropout_rate)
        self.dropout3 = Dropout(dropout_rate)

    def call(self, x, enc_output, training, look_ahead_mask, padding_mask):
        attn1 = self.mha1(x, x, x, mask=look_ahead_mask) 
        attn1 = self.dropout1(attn1, training=training)
        out1 = self.layernorm1(attn1 + x)

        attn2 = self.mha2(v=enc_output, k=enc_output, q=out1, mask=padding_mask)
        attn2 = self.dropout2(attn2, training=training)
        out2 = self.layernorm2(attn2 + out1)

        ffn_output = self.ffn(out2)
        ffn_output = self.dropout3(ffn_output, training=training)
        out3 = self.layernorm3(ffn_output + out2)

        return out3

## Decoder

In [143]:
class Decoder(tf.keras.layers.Layer):
    def __init__(self, num_layers, d_model, num_heads, dff, target_vocab_size, maximum_position_encoding, dropout_rate=0.1):
        super(Decoder, self).__init__()
        self.d_model = d_model
        self.num_layers = num_layers
        self.embedding = Embedding(target_vocab_size, d_model)
        self.pos_encoding = positional_encoding(
            maximum_position_encoding, d_model)
        self.dropout = Dropout(dropout_rate)
        self.dec_layers = [DecoderBlock(
            d_model, num_heads, dff, dropout_rate) for _ in range(num_layers)]

    def call(self, x, enc_output, training, look_ahead_mask, padding_mask):
        seq_len = tf.shape(x)[1]
        attention_weights = {}
        x = self.embedding(x)
        x += self.pos_encoding[:, :seq_len, :]
        x = self.dropout(x, training=training)
        # for i in range(self.num_layers):
        #     x = self.dec_layers[i](x, training=training, mask=look_ahead_mask)
        for layer in self.dec_layers:
            x = layer(x, enc_output=enc_output, training=training, look_ahead_mask=look_ahead_mask, padding_mask=padding_mask)
        return x, attention_weights

## Defining Transformer Model

In [144]:
class Transformer(tf.keras.Model):
    def __init__(self, num_layers, d_model, num_heads, dff,
                 input_vocab_size, target_vocab_size, maximum_position_encoding, dropout_rate=0.1):
        super(Transformer, self).__init__()
        self.encoder = Encoder(
            num_layers, 
            d_model, 
            num_heads, 
            dff,
            input_vocab_size, 
            maximum_position_encoding, 
            dropout_rate
        )
        self.decoder = Decoder(
            num_layers, 
            d_model, 
            num_heads, 
            dff,
            target_vocab_size, 
            maximum_position_encoding, 
            dropout_rate
        )
        self.final_layer = Dense(target_vocab_size)

    def call(self, inputs, training=False, look_ahead_mask=None, padding_mask=None):
        inp, tar = inputs
        enc_output = self.encoder(
            inp, 
            training=training, 
            mask=padding_mask
            )
        dec_output, _ = self.decoder(
            tar, 
            enc_output=enc_output, 
            training=training,
            look_ahead_mask=look_ahead_mask, 
            padding_mask=padding_mask
        )
        final_output = self.final_layer(dec_output)

        return final_output

## Training and testing the Model

In [145]:
# Defining Custom Parameters
num_layers = 4
d_model = 128
num_heads = 8
dff = 512
input_vocab_size = 8500
target_vocab_size = 8000
maximum_position_encoding = 10000
dropout_rate = 0.1

transformer = Transformer(
    num_layers,
    d_model,
    num_heads,
    dff,
    input_vocab_size,
    target_vocab_size,
    maximum_position_encoding,
    dropout_rate
)

inputs = tf.random.uniform(
    (64, 50), dtype=tf.int64, minval=0, maxval=input_vocab_size
    )
targets = tf.random.uniform(
    (64, 50), dtype=tf.int64, minval=0, maxval=target_vocab_size
    )

look_ahead_mask = None
padding_mask = None

output = transformer((inputs, targets), training=True,
                     look_ahead_mask=look_ahead_mask, padding_mask=padding_mask)
print(output.shape)

(64, 50, 8000)


## Translation Test

In [146]:
def create_padding_mask(seq):
    seq = tf.cast(tf.math.equal(seq, 0), tf.float32)
    return seq[:, tf.newaxis, tf.newaxis, :]  # (batch_size, 1, 1, seq_len)

def create_look_ahead_mask(size):
    mask = 1 - tf.linalg.band_part(tf.ones((size, size)), -1, 0)
    return mask  # (seq_len, seq_len)

# --- TEST CONFIGURATION ---
batch_size = 1
seq_len_in = 5   # e.g., "I love deep learning <pad>"
seq_len_out = 6  # e.g., "<start> I love deep learning <pad>"

# source: [12, 45, 7, 102, 0] 
sample_encoder_input = tf.constant([[12, 45, 7, 102, 0]], dtype=tf.int64)
# target: [1, 54, 89, 210, 22, 0]
sample_decoder_input = tf.constant([[1, 54, 89, 210, 22, 0]], dtype=tf.int64)

# Create masks
enc_padding_mask = create_padding_mask(sample_encoder_input)

look_ahead_mask = create_look_ahead_mask(tf.shape(sample_decoder_input)[1])
dec_padding_mask = create_padding_mask(sample_decoder_input)
combined_mask = tf.maximum(dec_padding_mask, look_ahead_mask)

# Run the inference
predictions = transformer(
    inputs=(sample_encoder_input, sample_decoder_input),
    training=False,
    look_ahead_mask=combined_mask,
    padding_mask=enc_padding_mask
)

print(f"Encoder Input (English) shape: {sample_encoder_input.shape}")
print(f"Decoder Input (French) shape: {sample_decoder_input.shape}")
print("-" * 30)
print(f"Transformer Output shape: {predictions.shape}") 
# Expected: (1, 6, 8000) -> (Batch, Target_Seq_Len, Target_Vocab_Size)

# Extract the most likely token IDs
predicted_ids = tf.argmax(predictions, axis=-1)
print(f"Predicted IDs for the translation: {predicted_ids.numpy()}")

Encoder Input (English) shape: (1, 5)
Decoder Input (French) shape: (1, 6)
------------------------------
Transformer Output shape: (1, 6, 8000)
Predicted IDs for the translation: [[5318 5318 5318 5318 5318 5318]]


In [147]:
# 1. Create a dummy vocabulary to match your 8000-word limit
# This maps every possible ID (0-7999) to a placeholder string
vocabulary = [f"token_{i}" for i in range(target_vocab_size)]

# 2. Get the IDs from your result
ids = predicted_ids.numpy()[0]

# 3. Convert IDs to "Text"
translated_text = [vocabulary[i] for i in ids]

print("-" * 30)
print(f"Predicted IDs: {ids}")
print(f"Translated Text: {' '.join(translated_text)}")

------------------------------
Predicted IDs: [5318 5318 5318 5318 5318 5318]
Translated Text: token_5318 token_5318 token_5318 token_5318 token_5318 token_5318
